 # Section 1: Setup, Unified Paths & Global Settings

 This cell loads essential utilities, configures global Pandas display parameters,

 silences silent-downcasting warnings, and dynamically maps project directories.

In [1]:
# 3. Path Selection Toggle

# USE_LOCAL_DATA = False  # Set False for GLOBAL_DATA_DIR
USE_LOCAL_DATA = True  # Set False for GLOBAL_DATA_DIR

In [2]:
%load_ext autoreload
%autoreload 2

import ast
from pathlib import Path

import numpy as np
import pandas as pd

from core.contracts import EngineInput, FilterPack
from core.paths import (
    GLOBAL_DATA_DIR,
    GLOBAL_PROCESSED_DIR,
    LOCAL_DATA_DIR,
    NOTEBOOKS_RLVR_ROOT,
    OUTPUT_DIR,
    PROJECT_ROOT,
)
from core.settings import TradingConfig
from core.utils import SystemUtils as SU
from data_pipeline import generate_features
from walk_forward import (
    AlphaEngine,
    create_walk_forward_analyzer,
    run_headless_simulation,
)

# 1. Global Configuration Options
config = TradingConfig()

# 2. Display Settings & Silence Warnings
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 3000)

# 3. Path Selection Toggle
# USE_LOCAL_DATA = False  # Set False for GLOBAL_DATA_DIR

data_dir = LOCAL_DATA_DIR if USE_LOCAL_DATA else GLOBAL_DATA_DIR
processed_dir = GLOBAL_PROCESSED_DIR
output_dir = OUTPUT_DIR

# 4. Define Target File Paths
ohlcv_filename = (
    "df_ohlcv.parquet" if USE_LOCAL_DATA else "df_OHLCV_stocks_etfs.parquet"
)
fed_filename = (
    "df_fed.parquet"
    if USE_LOCAL_DATA
    else "High_Yield_Spread_T10Y2Y_Spread.parquet"
)

raw_paths = {
    "df_ohlcv": data_dir / ohlcv_filename,
    "df_fed": data_dir / fed_filename,
    "df_indices": data_dir / "df_indices.parquet",
}

features_df_path = data_dir / "features_df.parquet"
macro_df_path = data_dir / "macro_df.parquet"

print(
    f"Paths Resolved:\n"
    f" - Active Mode:          {'LOCAL' if USE_LOCAL_DATA else 'GLOBAL'}\n"
    f" - Active Data Dir:       {data_dir}\n"
    f" - Global Processed Dir: {GLOBAL_PROCESSED_DIR}\n"
    f" - Output Dir:           {OUTPUT_DIR}\n"
)

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output

Paths Resolved:
 - Active Mode:          LOCAL
 - Active Data Dir:       C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
 - Global Processed Dir: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
 - Output Dir:           C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output



# Section 2: Ingest Datasets & Generate Features

Loads raw input files and retrieves pre-calculated features or runs feature generation.

In [3]:
# Load Raw Files
raw_dfs = {}

for name, path in raw_paths.items():
    if path.exists():
        raw_dfs[name] = pd.read_parquet(path)
        print(f"[OK] Successfully loaded {name:<10} from {path.name}")
    else:
        print(f"[ERROR] File not found: {path}")

df_ohlcv = raw_dfs.get("df_ohlcv")
df_fed = raw_dfs.get("df_fed")
df_indices = raw_dfs.get("df_indices")

[OK] Successfully loaded df_ohlcv   from df_ohlcv.parquet
[OK] Successfully loaded df_fed     from df_fed.parquet
[OK] Successfully loaded df_indices from df_indices.parquet


In [4]:
# Load Processed Features or Generate If Missing / Running Global
if USE_LOCAL_DATA and features_df_path.exists() and macro_df_path.exists():
    features_df = pd.read_parquet(features_df_path)
    macro_df = pd.read_parquet(macro_df_path)
    print(f"[OK] Loaded features_df from local disk | Shape: {features_df.shape}")
    print(f"[OK] Loaded macro_df from local disk    | Shape: {macro_df.shape}")
else:
    reason = (
        "USE_LOCAL_DATA is False"
        if not USE_LOCAL_DATA
        else "Local feature files missing"
    )
    print(f"[INFO] {reason}. Generating features (~6 mins)...")

    features_df, macro_df = generate_features(
        df_ohlcv=df_ohlcv, df_indices=df_indices, df_fed=df_fed, config=config
    )
    print(f"[OK] Generated features_df  | Shape: {features_df.shape}")
    print(f"[OK] Generated macro_df     | Shape: {macro_df.shape}")

[OK] Loaded features_df from local disk | Shape: (8570001, 22)
[OK] Loaded macro_df from local disk    | Shape: (7454, 10)


In [5]:
# Cache Processed DataFrames to GLOBAL_PROCESSED_DIR when running Global
if not USE_LOCAL_DATA:
    GLOBAL_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Saving DataFrames to: {GLOBAL_PROCESSED_DIR}\n")

    dfs_to_save = {
        "df_ohlcv.parquet": df_ohlcv,
        "df_indices.parquet": df_indices,
        "df_fed.parquet": df_fed,
        "features_df.parquet": features_df,
        "macro_df.parquet": macro_df,
    }

    for filename, df in dfs_to_save.items():
        if df is not None:
            save_path = GLOBAL_PROCESSED_DIR / filename
            df.to_parquet(save_path)
            print(f"[OK] Saved {filename:<20} | Shape: {df.shape}")
        else:
            print(f"[SKIP] {filename:<19} | DataFrame missing in memory")

In [6]:
###############################
###############################

In [7]:
print("🚀 Unstacking Wide Matrices...")

# 1. Unstack directly.
# (Since df_ohlcv is already a perfect grid of master_dates x tickers,
# unstacking automatically creates perfectly aligned wide dataframes).
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0)
df_trp_wide = features_df["TRP"].unstack(level=0)

print("Applying terminal NaN replacements for model inputs...")

# 2. Terminal Fills ONLY.
# Note: 0-to-NaN conversion and bounded ffills were ALREADY applied upstream to df_ohlcv.
# Applying ffill here would incorrectly double your max_data_gap_ffill limit!
df_close_wide = df_close_wide.fillna(config.nan_price_replacement)
df_atrp_wide = df_atrp_wide.fillna(0.0)
df_trp_wide = df_trp_wide.fillna(0.0)

print("All wide dataframes unstacked and prepared for downstream modeling.")
print(
    f"[OK] Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

🚀 Unstacking Wide Matrices...
Applying terminal NaN replacements for model inputs...
All wide dataframes unstacked and prepared for downstream modeling.
[OK] Pipeline Complete. Tickers: 1716, Days: 7454


 # Section 3: In-Memory Structural Previews

 Outputs head and tail previews of the core dataframes.

In [8]:
# print("\n" + "=" * 50)
# print("DATAFRAME STRUCTURAL PREVIEWS (HEAD & TAIL)")
# print("=" * 50)

# # 1. Preview wide matrix prices
# if "df_close_wide" in locals():
#     print("\n=== df_close_wide (Prices) ===")
#     print("--- HEAD (First 3 Rows) ---")
#     display(df_close_wide.head(3))
#     print("\n--- TAIL (Last 3 Rows) ---")
#     display(df_close_wide.tail(3))

# # 2. Preview multi-index features
# if "features_df" in locals():
#     print("\n=== features_df (Multi-Index Features) ===")
#     print("--- HEAD (First 3 Rows) ---")
#     display(features_df.head(3))
#     print("\n--- TAIL (Last 3 Rows) ---")
#     display(features_df.tail(3))

# # 3. Preview macro data
# if "macro_df" in locals():
#     print("\n=== macro_df (Macro Data) ===")
#     print("--- HEAD (First 3 Rows) ---")
#     display(macro_df.head(3))
#     print("\n--- TAIL (Last 3 Rows) ---")
#     display(macro_df.tail(3))

 # Section 4: Engine Initialization & Stage 1 Analysis

In [9]:
# Instantiate Master Engine
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("Master AlphaEngine Ready.")

Master AlphaEngine Ready.


In [10]:
# import pandas as pd

# target_date = pd.Timestamp("2025-09-01")

# # Use level=1 for the date level (since level names aren't 'timestamp')
# dates_in_index = df_ohlcv.index.get_level_values(1).normalize().unique()
# print(f"Labor Day 2025-09-01 in index: {target_date in dates_in_index}")

# # Check if date exists (one-liner)
# any_match = (df_ohlcv.index.get_level_values(1).normalize() == target_date).any()
# print(f"Any match: {any_match}")

# # If you want to see which tickers have data on that date:
# if any_match:
#     mask = df_ohlcv.index.get_level_values(1).normalize() == target_date
#     tickers_on_date = df_ohlcv.index.get_level_values(0)[mask].unique()
#     print(f"Tickers on 2025-09-01: {tickers_on_date}")
#     print(f"Count: {len(tickers_on_date)}")

# # Extract all data for that specific date
# if any_match:
#     # Using boolean mask (works regardless of index sorting)
#     mask = df_ohlcv.index.get_level_values(1).normalize() == target_date
#     df_target = df_ohlcv[mask]
#     print(df_target.head())

In [11]:
# Configuration for Interactive UI (Stage 1 Setup)
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-08-05"),
    lookback_period=189,
    holding_period=config.holding_period,
    metric="Sharpe (TRP)",
    benchmark_ticker=config.benchmark_ticker,
    rank_start=1,
    rank_end=50,  # Plot only display 50 tickers
    debug=False,
)

analyzer1, stage1_pack = create_walk_forward_analyzer(
    engine, _inputs, universe_subset=None
)
analyzer1.show()

In [12]:
# Execute Headless Agent Simulation
print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df.style.format("{:.6f}"))
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

--- 🤖 RL AGENT HEADLESS VIEW ---
----------------------------------------------------------------------
Timeline: [2025-10-31] -> Decision: 2026-08-05 -> Entry: 2026-08-06 -> End: 2026-08-13
Selected Tickers (50):
SGOV, SHV, BIL, VBIL, MINT, USFR, BOXX, PULS, JPST, JAAA
RVMD, RY, TD, PEN, SNDK, VLUE, SCHD, MU, JAZZ, IWD
ROST, NUE, HST, DELL, FNDX, CM, HSBC, WBS, ASX, RPRX
EWT, STT, VTRS, STM, VTV, CSCO, TRGP, STX, ATI, BNS
HPE, WDC, CASY, BUD, EFV, LITE, FTNT, SN, AMAT, TRV
----------------------------------------------------------------------


,Full,Lookback,Holding
Metric,,,
Group Gain,0.642649,0.617403,0.021140
Benchmark Gain,0.139705,0.129250,0.012054
== Gain Delta,0.502944,0.488152,0.009087
Group Sharpe,3.293997,3.243401,10.434500
Benchmark Sharpe,1.410333,1.334307,8.974761
== Sharpe Delta,1.883664,1.909093,1.459739
Group Sharpe (ATRP),0.104867,0.105447,0.138067
Benchmark Sharpe (ATRP),0.064430,0.061590,0.225413
== Sharpe (ATRP) Delta,0.040437,0.043857,-0.087345



Target Reward Signal: 0.021140


 # Section 5: Stage 2 - Cascade Filter Execution

In [13]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# LAUNCH STAGE 2 (Cascade on the Stage 1 subset)
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---
🚀 Ready for Stage 2: Run Simulation for 2nd filter.


In [14]:
# Parse and map results from Analyzer 2
result = SU.map_analyzer(analyzer2)

[SEARCH] ANALYZER SIMULATION MAP
[  0] [DATA] audit_pack (EngineOutput)
[  1]   [PLOT] portfolio_series (shape=(196,))
[  2]   [PLOT] benchmark_series (shape=(196,))
[  3]   [CALC] normalized_plot_data (shape=(196, 50))
[  4]   [FILE] tickers (len=50)
[  5]     [DOC] index_0 (str)
[  6]     [DOC] index_1 (str)
[  7]     [DOC] index_2 (str)
[  8]     [DOC] index_3 (str)
[  9]     [DOC] index_4 (str)
[ 10]     [DOC] index_5 (str)
[ 11]     [DOC] index_6 (str)
[ 12]     [DOC] index_7 (str)
[ 13]     [DOC] index_8 (str)
[ 14]     [DOC] index_9 (str)
[ 15]     [DOC] index_10 (str)
[ 16]     [DOC] index_11 (str)
[ 17]     [DOC] index_12 (str)
[ 18]     [DOC] index_13 (str)
[ 19]     [DOC] index_14 (str)
[ 20]     [DOC] index_15 (str)
[ 21]     [DOC] index_16 (str)
[ 22]     [DOC] index_17 (str)
[ 23]     [DOC] index_18 (str)
[ 24]     [DOC] index_19 (str)
[ 25]     [DOC] index_20 (str)
[ 26]     [DOC] index_21 (str)
[ 27]     [DOC] index_22 (str)
[ 28]     [DOC] index_23 (str)
[ 29]     [DOC

In [15]:
# Test different methods to extract resulting target tickers
_tickers = SU.fetch(4, result)
print(f"_tickers (by index): {_tickers}\n")

_tickers = SU.fetch("tickers", result)
print(f"_tickers (by key): {_tickers}\n")

print(f'result[4]["obj"]: {result[4]["obj"]}')

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'VBIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'JAAA',
 'RVMD',
 'RY',
 'TD',
 'PEN',
 'SNDK',
 'VLUE',
 'SCHD',
 'MU',
 'JAZZ',
 'IWD',
 'ROST',
 'NUE',
 'HST',
 'DELL',
 'FNDX',
 'CM',
 'HSBC',
 'WBS',
 'ASX',
 'RPRX',
 'EWT',
 'STT',
 'VTRS',
 'STM',
 'VTV',
 'CSCO',
 'TRGP',
 'STX',
 'ATI',
 'BNS',
 'HPE',
 'WDC',
 'CASY',
 'BUD',
 'EFV',
 'LITE',
 'FTNT',
 'SN',
 'AMAT',
 'TRV']

_tickers (by index): ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'VLUE', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'CM', 'HSBC', 'WBS', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'BNS', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV']

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'VBIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'JAAA',
 'RVMD',
 'RY',
 'TD',
 'PEN',
 'SNDK',
 'VLUE',
 'SCHD',
 'MU',
 'JAZZ',
 'IWD',
 'ROST',
 'NUE',
 'HST',
 'DELL',
 'FNDX',
 'CM',
 'HSBC',
 'WBS',
 'ASX',
 'RPRX',
 'EWT',
 'STT',
 'VTRS',
 'STM',
 'VTV',
 'CSCO',
 'TRGP',
 'STX',
 'ATI',
 'BNS',
 'HPE',
 'WDC',
 'CASY',
 'BUD',
 'EFV',
 'LITE',
 'FTNT',
 'SN',
 'AMAT',
 'TRV']

_tickers (by key): ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'VLUE', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'CM', 'HSBC', 'WBS', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'BNS', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV']

result[4]["obj"]: ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'VLUE', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'CM', 'HSBC', 'WBS', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'BNS', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV']


 # Section 6: Verification, Exports & Auditing

In [16]:
# Export analytical trace logs to Excel
SU.export_audit_to_excel(
    audit_pack=analyzer2.last_run, filename="Audit_Verification_Report.xlsx"
)

[FILE] [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\Audit_Verification_Report.xlsx
[DONE] Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/Audit_Verification_Report.xlsx')

In [17]:
# Export last-run stacked OHLCV ticker records
SU.export_last_run_tickers_data_to_csv(
    analyzer=analyzer2,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename="last_run_tickers_stacked.csv",
)

Creating combined dictionary for 51 ticker(s)
Date range: 2025-10-31 00:00:00 to 2026-08-13 00:00:00
Data retrieved for 51 ticker(s) from 2025-10-31 00:00:00 to 2026-08-13 00:00:00
Total rows: 9996
Date range in data: 2025-10-31 00:00:00 to 2026-08-13 00:00:00
  SGOV: 196 rows
  SHV: 196 rows
  BIL: 196 rows
  VBIL: 196 rows
  MINT: 196 rows
  USFR: 196 rows
  BOXX: 196 rows
  PULS: 196 rows
  JPST: 196 rows
  JAAA: 196 rows
  RVMD: 196 rows
  RY: 196 rows
  TD: 196 rows
  PEN: 196 rows
  SNDK: 196 rows
  VLUE: 196 rows
  SCHD: 196 rows
  MU: 196 rows
  JAZZ: 196 rows
  IWD: 196 rows
  ROST: 196 rows
  NUE: 196 rows
  HST: 196 rows
  DELL: 196 rows
  FNDX: 196 rows
  CM: 196 rows
  HSBC: 196 rows
  WBS: 196 rows
  ASX: 196 rows
  RPRX: 196 rows
  EWT: 196 rows
  STT: 196 rows
  VTRS: 196 rows
  STM: 196 rows
  VTV: 196 rows
  CSCO: 196 rows
  TRGP: 196 rows
  STX: 196 rows
  ATI: 196 rows
  BNS: 196 rows
  HPE: 196 rows
  WDC: 196 rows
  CASY: 196 rows
  BUD: 196 rows
  EFV: 196 rows
 

WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/last_run_tickers_stacked.csv')

 # Section 7: Slicing Tests & Dictionary Parsing

In [18]:
# Verify captured ticker names
print(f"_tickers: {_tickers}")

_tickers: ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'VLUE', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'CM', 'HSBC', 'WBS', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'BNS', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV']


In [19]:
# Test various pandas multi-index slicing configurations
##############
test_tickers = ["NVDA", "GOOG"]
##############

idx = pd.IndexSlice
columns = ["Adj Close", "Volume"]

# # Slice 1: All dates, all columns
# display(df_ohlcv.loc[idx[test_tickers]].copy().head(3))

# # Slice 2: specific start date, all columns
# display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], :].head(3))

# # Slice 3: specific start date, selected columns
# display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], columns].head(3))

# Slice 4: specific start/end range, all columns
display(
    df_ohlcv.loc[
        idx[test_tickers, pd.Timestamp("2026-08-07") : pd.Timestamp("2026-08-14")], :
    ]
)

Adj Open  Adj High  Adj Low  Adj Close       Volume
Ticker Date                                                           
NVDA   2026-08-07    221.54    224.76   220.66     223.96  105669400.0
       2026-08-10    223.40    224.14   216.77     217.55  115846600.0
       2026-08-11    222.17    222.20   216.20     217.50  101273100.0
       2026-08-12    221.04    225.10   220.20     224.09  108783600.0
       2026-08-13    225.06    227.23   223.71     225.30   98867200.0
       2026-08-14    226.77    227.49   224.50     225.16   75680900.0
GOOG   2026-08-07    355.28    357.55   352.82     353.47   14516700.0
       2026-08-10    354.24    355.93   351.47     355.84   12789500.0
       2026-08-11    354.00    355.00   342.57     343.00   18390200.0
       2026-08-12    345.33    345.50   339.82     342.37   15608800.0
       2026-08-13    344.57    345.83   342.40     343.94   14997800.0
       2026-08-14    344.26    348.06   342.07     343.54   10946800.0

In [20]:
# Test custom combined dictionary extraction
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=_tickers,
    date_start="2026-05-01",
    date_end=None,
    verbose=False,
)

{'SGOV':             Adj Open  Adj High   Adj Low  Adj Close      Volume       ATR      ATRP       TRP       RSI    Mom_21  Consistency     IR_63   Beta_63  DD_21  AutoCorr_15    Ret_1d  Range_Pos_20  Slope_P_5  Slope_V_5  Slope_P_5_Z  Slope_V_5_Z     Convexity  RollingStalePct  RollMedDollarVol  RollingSameVolCount  RecentStaleDays  IsZeroPrice
 Date                                                                                                                                                                                                                                                                                                                                              
 2026-05-01   99.5131   99.5131   99.5032    99.5131  34114411.0  0.019316  0.000194  0.000378  0.999997  0.003168          0.6 -0.061534 -0.003309    0.0    -0.573851  0.000378      1.000000   0.000145   0.739754     0.085094    -0.303915  6.087777e-06         0.011905      1.368668e+09                  0.0   

 # Section 8: Finviz Ingestion & Filtering

In [21]:
# Load latest updated Finviz dataset dynamically
def load_latest_finviz_parquet(data_dir: Path) -> pd.DataFrame:
    pattern = "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]_df_finviz_merged_stocks_etfs.parquet"
    files = list(data_dir.glob(pattern))

    if not files:
        print(f"DEBUG: No files matching pattern found in {data_dir.absolute()}")
        return None

    latest_path = max(files, key=lambda x: x.name)
    print(f"DEBUG: Loading file: {latest_path.name}")

    try:
        return pd.read_parquet(latest_path)
    except Exception as e:
        print(f"DEBUG: Failed to read {latest_path.name}. Error: {e}")
        return None


# Load the file using the unified data_dir path
df_finviz = load_latest_finviz_parquet(GLOBAL_DATA_DIR)
print(
    f"df_finviz raw records shape: {df_finviz.shape if df_finviz is not None else None}"
)

DEBUG: Loading file: 2026-08-17_df_finviz_merged_stocks_etfs.parquet
df_finviz raw records shape: (1150, 139)


In [22]:
display(df_finviz.head())
display(df_finviz.tail())

,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
NVDA,1,NVIDIA Corp,"DJIA, NDX, S&P 500",Technology,Semiconductors,USA,NASD,"Technology, Semiconductors",5445240.0,1,5445240.0,34.46,NaN,0.37,21.48,27.88,67.58,45.73,8.07,3.33,0.29,0.28,6/4/2026,0.82,6.53,2.08,NaN,NaN,NaN,NaN,NaN,85.23,213.42,109.57,70.68,253490.0,159610.0,6.52,3.43,24220.0,23280.0,96.12,3.85,-0.30,70.83,0.00,1.26,2.00,292.67,82.97,114.29,77.17,3.44,2.85,0.06,0.07,74.15,64.02,62.97,-3.375378,3.43,10.95,-0.14,23.08,24.69,20.65,2.22,6.65,2.955424,1.85,3.04,6.40,8.89,15.37,-1.09,18.55,-4.87,37.14,164.07 - 236.54,-4.87,674929.95,62.85,Aug 26/a,1/22/1999,Yes,Yes,42000.0,-0.43,0.36,1.23,146.53,0.64,93678685,314.29,225.16,225.98,227.92,224.86,225.01,-0.07,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-402.163305,-15.868328,0.000000,9.092678,236.121822,22.043596,5.438386,11.035632,2.880046,6.840089,12.441182,2.988116,2.992416,4.749957,1.596061,0.373800,0.540906,1.061105,0.853946,1.255822,1.145043,NaN,NaN,NaN
AAPL,2,Apple Inc,"DJIA, NDX, S&P 500",Technology,Consumer Electronics,USA,NASD,"Technology, Consumer Electronics",4459840.0,2,4459840.0,35.03,NaN,2.55,9.55,41.52,71.47,32.63,7.36,4.28,0.35,1.06,8/10/2026,13.66,8.72,1.98,NaN,NaN,NaN,NaN,NaN,16.36,29.13,32.57,14.24,466820.0,128930.0,6.77,0.35,14610.0,14580.0,99.77,0.12,-2.06,67.76,0.00,0.97,2.52,141.61,36.08,148.75,72.08,1.00,0.93,0.66,0.78,48.65,33.17,27.62,3.562907,-0.87,-8.43,1.79,19.47,31.95,12.41,1.07,7.64,2.500082,1.60,2.11,-3.71,-1.13,8.88,-11.31,11.63,-11.31,36.56,223.78 - 344.57,-11.31,480266.51,43.41,Jul 30/a,12/12/1980,Yes,Yes,166000.0,-0.20,0.09,2.04,56.16,0.68,38169263,334.23,305.93,306.21,307.66,302.94,305.59,-0.11,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,2.600739,6.770046,1.603124,0.862471,1.489024,1.165800,-2.614915,-3.169573,0.656557,-5.444573,-5.538972,0.293815,-0.394592,-0.503117,0.927599,0.075713,0.100695,1.013565,0.835821,1.168888,1.157265,NaN,NaN,NaN
GOOGL,3,Alphabet Inc,"DJIA, NDX, S&P 500",Communication Services,Internet Content & Information,USA,NASD,"Communication Services, Internet Content & Inf...",4194150.0,3,4194150.0,17.28,NaN,1.26,9.40,6.76,17.30,78.73,50.90,19.89,0.23,0.85,9/4/2026,7.68,19.91,3.02,NaN,NaN,NaN,NaN,NaN,23.98,294.02,112.71,20.23,446310.0,244120.0,216.61,2.88,5870.0,5850.0,99.62,52.20,-0.01,39.24,0.00,1.21,2.16,70.49,34.30,48.68,32.33,2.72,2.64,0.18,0.18,60.94,33.96,54.70,-0.341139,-3.78,-0.80,-13.30,12.52,68.71,9.90,1.22,10.60,3.081395,2.01,2.73,-0.58,-2.64,3.69,-10.53,9.24,-15.81,74.98,196.60 - 408.61,-15.81,14225.43,46.03,Jul 22/a,8/19/2004,Yes,Yes,190820.0,-0.65,0.11,1.30,3

,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
DSI,383,iShares ESG MSCI KLD 400 ETF,-,Financial,Exchange Traded Fund,USA,NYSE,"Financial, Exchange Traded Fund, US Equities -...",5570.0,1543,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.84,1.23,6/15/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.49,0.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.267017,0.27,5.56,4.72,14.42,21.76,14.42,1.08,1.60,1.085408,0.55,0.98,2.85,3.88,10.66,-0.91,8.31,-0.91,25.91,117.08 - 148.76,-0.91,1083.54,62.97,-,11/17/2006,Yes,Yes,NaN,-0.49,-0.17,NaN,0.11084,0.52,57322,NaN,148.38,148.13,148.34,147.40,147.41,-0.65,US Equities - Factor & Thematic,Equities (Stocks),0.25,406.0,5570.0,NaN,-0.14,20.34,0.37,-269.21,-4.61,23.06,21.88,12.86,"U.S., equity, ESG",-11.305467,-11.265004,0.000000,3.491990,6.417560,1.805663,0.254245,0.421223,1.040274,6.420027,16.421413,3.035439,2.056706,3.529952,1.427784,1.058180,1.560791,1.190264,1.470383,2.261392,1.271005,NaN,NaN,NaN
EFAV,386,iShares MSCI EAFE Min Vol Factor ETF,-,Financial,Exchange Traded Fund,USA,CBOE,"Financial, Exchange Traded Fund, Global or ExU...",5480.0,1545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.07,2.86,6/15/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.60,0.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.427077,-0.49,2.34,3.25,-0.68,9.19,8.03,0.48,0.75,0.804894,0.43,0.58,0.59,3.04,3.96,-1.17,7.55,-2.04,11.77,83.36 - 95.12,-2.04,96.17,58.57,-,10/20/2011,Yes,Yes,NaN,-0.20,-0.31,NaN,0.51894,0.85,443171,NaN,93.66,93.37,93.44,93.01,93.18,-0.51,Global or ExUS Equities - Factor & Thematic,Equities (Stocks),0.20,278.0,5480.0,NaN,1.89,-23.43,-0.43,-126.77,-2.26,13.56,14.99,6.96,"EAFE, equity, volatility",-3.186390,-4.963741,0.557795,-4.491514,-5.574978,0.449698,1.294388,2.415360,1.244470,0.944475,1.511201,1.166681,3.110626,5.288274,1.666441,0.902965,1.437857,1.162315,-0.141445,-0.197703,0.976314,NaN,NaN,NaN
FDN,390,First Trust Dow Jones Internet Index Fund,-,Financial,Exchange Traded Fund,USA,NYSE,"Financial, Exchange Traded Fund, US Equities -...",5410.0,1550,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12/21/2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.10,0.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.746860,-2.11,6.37,7.00,21.23,5.43,6.60,1.22,5.50,1.916777,1.47,1.71,2.96,5.43,9.06,-2.92,12.97,-2.92,27.83,224.46 - 295.57,-2.92,2584.19,57.58,-,6/23/2006,Yes,Yes,NaN,-1.17,-0.36,NaN,0.38555,0.39,150458,NaN,291.38,290.34,290.73,286.85,286.94,-1.52,US Equities - Indu

In [23]:
raw_text = """
SMTC, TER, MTSI, PRAX, TTMI, LSCC,
"""

# Split by comma, strip whitespace, and filter out any empty strings
ticker_list = [item.strip() for item in raw_text.split(",") if item.strip()]

print(ticker_list)

['SMTC', 'TER', 'MTSI', 'PRAX', 'TTMI', 'LSCC']


In [24]:
# Set test target tickers
##################
# target_tickers = ["GLW", "NOK", "DELL", "NVDA", "GOOG"]
target_tickers = ticker_list

print(f"target_tickers: {target_tickers}")
##################

target_tickers: ['SMTC', 'TER', 'MTSI', 'PRAX', 'TTMI', 'LSCC']


In [25]:
# Clean index extraction using ast exception parsing if tickers are missing
try:
    result = df_finviz.loc[target_tickers]
except KeyError as e:
    error_msg = str(e)
    missing_str = error_msg.replace(" not in index", "").strip('"').strip("'")
    missing_tickers = ast.literal_eval(missing_str)

    print(f"Missing tickers: {missing_tickers}")

    # Strip out the missing symbols and retry slicing
    target_tickers_in_finviz = [t for t in target_tickers if t not in missing_tickers]
    result = df_finviz.loc[target_tickers_in_finviz].copy()
    result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

print(f"result shape: {result.shape}")

Missing tickers: ['MTSI', 'PRAX']
result shape: (4, 139)


In [26]:
# Display filtered Finviz information and schema details
result.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4 entries, TER to SMTC
Columns: 139 entries, No. to Omega 250d
dtypes: float64(120), int64(3), object(16)
memory usage: 4.4+ KB


In [27]:
# Check specific target column (Market Capitalization)
result["MktCap AUM, M"]

TER     69280.0
LSCC    18840.0
TTMI    14850.0
SMTC    14370.0
Name: MktCap AUM, M, dtype: float64

In [28]:
# Sort records descendingly by Market Capitalization
result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

In [29]:
# View final sorted DataFrame
print(result)

      No.                     Company         Index      Sector                             Industry Country Exchange                                             Info  MktCap AUM, M  Rank  Market Cap, M     P/E  Fwd P/E   PEG    P/S    P/B     P/C   P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %   EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %  ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Price %  Volatility W %  Volatility M %  SMA20 %  SMA50 %  SMA200 %  50D High %  50D Low %  52W High %  52W Low %        

In [30]:
# # Export processed dataframe output directly to trash file
# result.to_csv(output_dir / "_trash.csv")